# Trim (clipping)

**Kerakli fayllar:** `trim_utils.py`, `tinglash_royxati.csv` (qo'lda tinglash qarorlari), `hf_cache/`

**Natija:** `DATASET_trimmed/{split}/{idx}.flac`, `chiqarilganlar.csv`, `trim_manifest_yakuniy.csv`

| Qadam | Nima qiladi |
|---|---|
| 1 | Kutubxonalar va sozlamalar |
| 2 | Dataset va Silero modelini yuklash |
| 3 | Ko'p jarayonli ishga tushirish funksiyasi |
| 4 | Butun datasetni trim qilish (~20 daqiqa, tayyor bo'lsa darhol tugaydi) |
| 5 | Manifestni o'qish va shubhalilarni belgilash |
| 6 | Qo'lda tinglash qarorlarini o'qish |
| 7 | Qo'lda tasdiqlangan audiolarni qayta kesish |
| 8 | Chiqariladigan audiolar ro'yxati |
| 9 | Yakuniy manifest |

In [1]:
# 1-QADAM. Kutubxonalar va sozlamalar

import importlib                  # trim_utils.py o'zgarsa, uni qayta yuklash uchun
from pathlib import Path          # fayl yo'llari bilan qulay ishlash uchun
from multiprocessing import Pool  # bir vaqtda bir nechta jarayonda ishlash uchun
import time                       # vaqtni o'lchash uchun

import numpy as np
import pandas as pd
import soundfile as sf            # audio faylni o'qish va yozish
from tqdm.auto import tqdm        # progress chizig'i

import trim_utils                 # bizning funksiyalarimiz: audio_oqish, trim_qil, ...
importlib.reload(trim_utils)

# Trim parametrlari trim_utils.py ichida turadi:
# THRESHOLD = 0.4, PAD_MS = 200, MIN_SPEECH_MS = 250, MIN_ULUSH = 0.7, MAX_CPS = 24
SR = trim_utils.SR                 # 16000 Hz
PAD_MS = trim_utils.PAD_MS         # nutq atrofida qoldiriladigan zaxira (ms)

JARAYONLAR = 11                                # 12 yadrodan 1 tasi kompyuter uchun bo'sh qoladi
CACHE = str(Path("../hf_cache").resolve())      # dataset keshining to'liq yo'li

CHIQISH_PAPKA = Path("../DATASET_trimmed")     # kesilgan audiolar loyiha ildizidagi shu papkaga yoziladi
XOM_MANIFEST = Path("trim_manifest.csv")       # har bir audio haqida trim ma'lumoti (oraliq fayl, 4-qadam qayta yaratadi)
QARORLAR_FAYL = Path("tinglash_royxati.csv")   # qo'lda tinglash natijalari
CHIQARILGAN_FAYL = Path("chiqarilganlar.csv")
YAKUNIY_MANIFEST = Path("trim_manifest_yakuniy.csv")

<python>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2-QADAM. Dataset va Silero modelini yuklash

# Bu funksiya trim_utils ichida datasetni (_ds) va modelni (_model) yuklaydi.
# Ishchi jarayonlar uni o'zlari chaqiradi. Bu yerda asosiy jarayon uchun chaqiramiz,
# chunki 7-qadamda ba'zi audiolarni qayta o'qiymiz.
trim_utils.ishchi_boshlash(CACHE)
ds = trim_utils._ds

# Barcha audiolar ro'yxatini (split, idx) juftliklari ko'rinishida yig'amiz.
# Tashqi for: har bir split ("train", "validation", "test") uchun.
# Ichki for: shu splitdagi har bir audio raqami (0, 1, 2, ...) uchun.
barcha = []
for split in ds:
    for idx in range(len(ds[split])):
        barcha.append((split, idx))

print("Jami audio:", len(barcha))

Jami audio: 21702


In [3]:
# 3-QADAM. Ko'p jarayonli ishga tushirish funksiyalari

def manifestga_yoz(natijalar, fayl):
    """Natijalarni CSV oxiriga qo'shadi. Sarlavha faqat fayl yangi bo'lsa yoziladi."""
    jadval = pd.DataFrame(natijalar)
    jadval = jadval.reindex(columns=trim_utils.USTUNLAR)   # ustunlar tartibi doim bir xil
    jadval.to_csv(fayl, mode="a", header=not fayl.exists(), index=False)


def ishga_tushir(vazifalar, chiqish_papka, manifest_fayl, saqlash_har=500):
    """Audiolarni 11 ta jarayonda trim qiladi.
    To'xtab qolsa, qayta chaqirilganda faqat tugallanmaganlarini bajaradi."""
    manifest_fayl = Path(manifest_fayl)

    # 1) Avval qaysi audiolar tayyor ekanini aniqlaymiz (xatosizlari)
    tayyor = set()
    if manifest_fayl.exists():
        eski = pd.read_csv(manifest_fayl, usecols=["split", "idx", "xato"])
        eski = eski[eski["xato"].isna()]
        tayyor = set(zip(eski["split"], eski["idx"]))

    # 2) Faqat tayyor bo'lmaganlarini qoldiramiz
    qolgan = []
    for split, idx in vazifalar:
        if (split, idx) not in tayyor:
            qolgan.append((split, idx, str(chiqish_papka)))

    print(f"Jami: {len(vazifalar)}, tayyor: {len(tayyor)}, qolgan: {len(qolgan)}")
    if len(qolgan) == 0:
        return

    # 3) Ishni jarayonlarga bo'lib beramiz.
    # initializer: har bir jarayon ishga tushganda dataset va modelni BIR MARTA yuklaydi.
    # imap_unordered: qaysi audio birinchi tayyor bo'lsa, natijasi shu zahoti keladi.
    # chunksize=8: har jarayonga vazifalar 8 tadan beriladi (aloqa xarajati kamayadi).
    bufer = []
    boshlanish = time.time()
    with Pool(JARAYONLAR, initializer=trim_utils.ishchi_boshlash, initargs=(CACHE,)) as pool:
        natijalar = pool.imap_unordered(trim_utils.bitta_ishlov, qolgan, chunksize=8)
        for natija in tqdm(natijalar, total=len(qolgan)):
            bufer.append(natija)
            # Har 500 ta natijada diskka yozamiz: kompyuter o'chib qolsa ham ish yo'qolmaydi
            if len(bufer) >= saqlash_har:
                manifestga_yoz(bufer, manifest_fayl)
                bufer = []

    # 4) Oxirgi qoldiqni ham yozamiz
    if len(bufer) > 0:
        manifestga_yoz(bufer, manifest_fayl)

    vaqt = time.time() - boshlanish
    print(f"Vaqt: {vaqt / 60:.1f} daqiqa, bir audio: {vaqt / len(qolgan):.3f} s")

In [4]:
# 4-QADAM. Butun datasetni trim qilish
# Birinchi marta ~20 daqiqa ishlaydi. Manifest allaqachon to'liq bo'lsa, "qolgan: 0" deb darhol tugaydi.
ishga_tushir(barcha, CHIQISH_PAPKA, XOM_MANIFEST)

Jami: 21702, tayyor: 0, qolgan: 21702


100%|██████████| 21702/21702 [17:29<00:00, 20.68it/s] 


Vaqt: 17.5 daqiqa, bir audio: 0.048 s


In [5]:
# 5-QADAM. Manifestni o'qish va shubhalilarni belgilash

m = pd.read_csv(XOM_MANIFEST)
m = m.drop_duplicates(["split", "idx"], keep="last")   # bitta audio bir marta qolsin
print("Audiolar:", len(m), "| Xatolilar:", m["xato"].notna().sum())

# CSV'dagi bo'sh qiymat NaN bo'lib o'qiladi. Solishtirish oson bo'lishi uchun "" ga aylantiramiz.
m["rad_sababi"] = m["rad_sababi"].fillna("")

# CPS = soniyadagi belgilar soni (normal o'rtacha ~16)
m["cps"] = m["matn"].str.len() / m["yangi_s"]

# Bayroqlar (True/False ustunlar)
m["b_kop_kesilgan"] = (m["kesilgan_bosh"] > 1.5) | (m["kesilgan_oxir"] > 1.5)
m["b_cps"] = m["cps"] > 24
m["b_rad"] = m["rad_sababi"] != ""
m["shubhali"] = m["b_kop_kesilgan"] | m["b_cps"] | m["nutqsiz"] | m["b_rad"]

print(f"Davomiylik: {m['eski_s'].sum() / 3600:.2f} soat -> {m['yangi_s'].sum() / 3600:.2f} soat")
print(m[["b_kop_kesilgan", "b_cps", "nutqsiz", "b_rad", "shubhali"]].sum())

Audiolar: 21702 | Xatolilar: 0
Davomiylik: 106.92 soat -> 104.71 soat
b_kop_kesilgan    512
b_cps             110
nutqsiz            13
b_rad             119
shubhali          753
dtype: int64


In [6]:
# 6-QADAM. Qo'lda tinglash qarorlarini o'qish
# tinglash_royxati.csv ichida har bir tinglangan audio uchun "qaror" ustuni bor:
#   ok         -> qoladi
#   chiqar     -> datasetdan chiqariladi
#   qayta_trim -> himoya rad etgan, lekin tinglab tasdiqlangan: saqlangan chegaralardan kesiladi
r = pd.read_csv(QARORLAR_FAYL)
print(r.groupby(["guruh", "qaror"]).size())


def juftliklar(jadval):
    """Jadvaldagi (split, idx) juftliklarini to'plam (set) qilib qaytaradi."""
    return set(zip(jadval["split"], jadval["idx"]))


qayta_trim = juftliklar(r[r["qaror"] == "qayta_trim"])
qolda_ok = juftliklar(r[r["qaror"] == "ok"])
print("\nQayta kesiladi:", len(qayta_trim), "| Tinglab tasdiqlangan:", len(qolda_ok))

guruh            qaror     
1_nutqsiz        chiqar        13
2_rad_qisqargan  chiqar        29
                 qayta_trim    13
3_valtest        chiqar        28
                 ok            38
4_rad_cps        chiqar         7
                 qayta_trim     1
5_kop_kesilgan   chiqar         1
                 ok            11
6_cps_imzosiz    chiqar         7
dtype: int64

Qayta kesiladi: 14 | Tinglab tasdiqlangan: 49


In [7]:
# 7-QADAM. Qo'lda tasdiqlangan audiolarni qayta kesish
# Silero topgan nutq chegaralari (nutq_bosh_s, nutq_oxir_s) manifestda saqlangan,
# shuning uchun VAD qayta ishlatilmaydi.

for split, idx in sorted(qayta_trim):
    # Shu audioning manifestdagi qatori
    i = m.index[(m["split"] == split) & (m["idx"] == idx)][0]
    eski_s = m.loc[i, "eski_s"]

    # Nutq chegarasiga 200 ms zaxira qo'shamiz, audio tashqarisiga chiqib ketmasin
    bosh = max(0.0, m.loc[i, "nutq_bosh_s"] - PAD_MS / 1000)
    oxir = min(eski_s, m.loc[i, "nutq_oxir_s"] + PAD_MS / 1000)

    # Asl audioni o'qib, kesamiz (soniya -> namuna raqami: soniya * 16000)
    wav, _ = trim_utils.audio_oqish(split, idx)
    kesilgan = wav[int(bosh * SR):int(oxir * SR)]
    kesilgan = np.clip(kesilgan, -1.0, 1.0)

    # DATASET_trimmed dagi kesilmagan nusxa ustiga yozamiz (asl dataset o'zgarmaydi)
    sf.write(CHIQISH_PAPKA / split / f"{idx}.flac", kesilgan, SR, format="FLAC", subtype="PCM_16")

    # Manifestni yangilaymiz
    m.loc[i, "yangi_s"] = len(kesilgan) / SR
    m.loc[i, "start_s"] = bosh
    m.loc[i, "end_s"] = oxir
    m.loc[i, "kesilgan_bosh"] = bosh
    m.loc[i, "kesilgan_oxir"] = eski_s - oxir
    m.loc[i, "rad_sababi"] = "qolda_trim"
    print(f"{split} #{idx}: {eski_s:.2f}s -> {len(kesilgan) / SR:.2f}s")

test #605: 15.55s -> 8.62s
test #1050: 19.11s -> 11.76s
train #1788: 23.11s -> 16.11s
train #3004: 12.73s -> 8.75s
train #6894: 22.33s -> 11.91s
train #7018: 15.37s -> 10.68s
train #7031: 14.15s -> 9.90s
train #7887: 15.31s -> 7.82s
train #8304: 15.83s -> 10.42s
train #9321: 10.53s -> 10.03s
train #12751: 14.07s -> 9.01s
train #16174: 28.35s -> 19.38s
train #16208: 30.23s -> 18.93s
train #18614: 28.23s -> 14.35s


In [8]:
# 8-QADAM. Chiqariladigan audiolar ro'yxati

# a) Tinglab "chiqar" deyilganlar
chiqar = r[r["qaror"] == "chiqar"][["split", "idx", "guruh"]]
chiqar = chiqar.rename(columns={"guruh": "sabab"})

# b) Umumiy qoida: namunalarning deyarli hammasi yaroqsiz chiqqan guruhlar
#    - himoya faqat CPS sababli rad etganlar (rad_sababi == "cps")
#    - CPS > 24 bo'lganlar
#    Tinglab tasdiqlanganlar bu qoidaga tushmaydi.
himoyalangan = qayta_trim | qolda_ok
umumiy = m[(m["rad_sababi"] == "cps") | m["b_cps"]]

saqlansin = []
for split, idx in zip(umumiy["split"], umumiy["idx"]):
    saqlansin.append((split, idx) not in himoyalangan)
umumiy = umumiy[saqlansin]

qoshimcha = pd.DataFrame({
    "split": umumiy["split"],
    "idx": umumiy["idx"],
    "sabab": np.where(umumiy["rad_sababi"] == "cps", "rad_cps_umumiy", "cps_umumiy"),
})

# a) va b) ni birlashtiramiz; bitta audio ikki marta bo'lsa, birinchisi (qo'lda sabab) qoladi
chiqar = pd.concat([chiqar, qoshimcha]).drop_duplicates(["split", "idx"])
chiqar.to_csv(CHIQARILGAN_FAYL, index=False)

print(chiqar["sabab"].value_counts())
print("Jami chiqariladi:", len(chiqar))

sabab
cps_umumiy         86
rad_cps_umumiy     61
2_rad_qisqargan    29
3_valtest          28
1_nutqsiz          13
4_rad_cps           7
6_cps_imzosiz       7
5_kop_kesilgan      1
Name: count, dtype: int64
Jami chiqariladi: 232


Natija: ~232 ta audio. Fayllar o'chirilmaydi, ular faqat yakuniy manifestga kirmaydi.

In [9]:
# 9-QADAM. Yakuniy manifest

# CPS ni qayta kesilganlar uchun yangilaymiz
m["cps"] = m["matn"].str.len() / m["yangi_s"]

# Chiqarilmaganlarni qoldiramiz
chiqarilganlar = juftliklar(chiqar)
saqlansin = []
for split, idx in zip(m["split"], m["idx"]):
    saqlansin.append((split, idx) not in chiqarilganlar)
yakuniy = m[saqlansin].copy()

# O'qitish uchun har bir audioning FLAC fayliga yo'l
yakuniy["fayl"] = [str(Path("DATASET_trimmed") / s / f"{i}.flac") for s, i in zip(yakuniy["split"], yakuniy["idx"])]

# Tekshiruv: hamma fayl diskda bormi?
topilmadi = [f for f in yakuniy["fayl"] if not (Path("..") / f).exists()]   # fayl yo'li loyiha ildiziga nisbatan
print("Topilmagan fayllar:", len(topilmadi))   # 0 bo'lishi kerak

yakuniy.to_csv(YAKUNIY_MANIFEST, index=False)

print("Qoldi:", len(yakuniy))
print(f"Soat: {yakuniy['yangi_s'].sum() / 3600:.2f}")
print(yakuniy.groupby("split").size())

Topilmagan fayllar: 0
Qoldi: 21470
Soat: 103.89
split
test           1056
train         19350
validation     1064
dtype: int64


Natija: ~21470 audio, ~103.9 soat, topilmagan fayllar 0.